# Star Schema: Dimensões e Fato

Este notebook implementa a modelagem star schema na camada **gold** do catálogo `capgemini_trainingforthecase`.

**Tabelas criadas:**
- `gold.dim_customer` — dimensão de clientes
- `gold.dim_product` — dimensão de produtos
- `gold.dim_date` — dimensão temporal (calendário)
- `gold.fact_sales` — tabela fato de vendas (ligada às dimensões via surrogate keys)

**Validações:** chaves naturais únicas, contagem de linhas antes/depois dos joins, clientes/produtos sem correspondência, receita antes/depois da modelagem.

**Summaries refeitos com PySpark:** `gold.customer_summary_v2` e `gold.product_summary_v2`.

In [0]:
# ============================================================
# Configuração — catalog e schema da camada gold
# ============================================================

catalog = "capgemini_trainingforthecase"
schema  = "gold"

print(f"Catalog: {catalog}")
print(f"Schema:  {schema}")
print(f"Base:    {catalog}.{schema}")

In [0]:
# ============================================================
# 1. DIM_CUSTOMER
#    Surrogate key (customer_sk) + natural key (customer_id)
#    Fonte: silver.customers
# ============================================================

from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

df_dim_customer = (
    spark.table(f"{catalog}.silver.customers")
    .select(
        col("customer_id"),
        col("customer_name"),
        col("gender"),
        col("age"),
        col("age_group"),
        col("date_of_birth"),
        col("email"),
        col("phone"),
        col("city"),
        col("state"),
        col("pincode"),
        col("registration_date"),
        col("customer_tier"),
    )
    # adiciona surrogate key sequencial
    .withColumn("customer_sk", row_number().over(Window.orderBy("customer_id")))
)

(
    df_dim_customer.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_customer")
)

print(f"dim_customer criada: {spark.table(f'{catalog}.{schema}.dim_customer').count()} linhas")
spark.table(f"{catalog}.{schema}.dim_customer").limit(3).display()

In [0]:
# ============================================================
# 2. DIM_PRODUCT
#    Surrogate key (product_sk) + natural key (product_id)
#    Fonte: silver.products
# ============================================================

df_dim_product = (
    spark.table(f"{catalog}.silver.products")
    .select(
        col("product_id"),
        col("product_name"),
        col("category"),
        col("brand"),
        col("original_price"),
        col("discount_percent"),
        col("discount_amount"),
        col("selling_price"),
        col("stock_quantity"),
        col("weight_kg"),
        col("avg_rating"),
        col("total_reviews"),
    )
    .withColumn("product_sk", row_number().over(Window.orderBy("product_id")))
)

(
    df_dim_product.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_product")
)

print(f"dim_product criada: {spark.table(f'{catalog}.{schema}.dim_product').count()} linhas")
spark.table(f"{catalog}.{schema}.dim_product").limit(3).display()

In [0]:
# ============================================================
# 3. DIM_DATE
#    Calendário completo cobrindo todas as datas de order_date
#    date_sk no formato YYYYMMDD (surrogate key semântica)
# ============================================================

from pyspark.sql.functions import (
    min as spark_min, max as spark_max, date_format, 
    year, quarter, month, dayofweek, dayofmonth, weekofyear,
    when, lit, col as scol
)

# Descobrir range de datas na tabela de vendas
date_bounds = spark.table(f"{catalog}.silver.sales").select(
    spark_min("order_date").alias("min_date"),
    spark_max("order_date").alias("max_date"),
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]
print(f"Range de datas: {min_date} a {max_date}")

# Gerar calendário via Spark range
days_diff = (max_date - min_date).days + 1

from pyspark.sql.functions import date_add

df_dim_date = (
    spark.range(days_diff)
    .select(date_add(lit(min_date), col("id").cast("int")).alias("date"))
    .withColumn("date_sk", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("day_of_month", dayofmonth(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
    .withColumn("week_of_year", weekofyear(col("date")))
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), lit(True)).otherwise(lit(False)))
    .withColumn("is_month_start", when(col("day_of_month") == 1, lit(True)).otherwise(lit(False)))
    .withColumn("is_month_end", lit(False))  # placeholder, corrigido abaixo
)

from pyspark.sql.functions import last_day

df_dim_date = (
    df_dim_date.drop("is_month_end")
    .withColumn("is_month_end", when(col("date") == last_day(col("date")), lit(True)).otherwise(lit(False)))
)

(
    df_dim_date.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_date")
)

print(f"dim_date criada: {spark.table(f'{catalog}.{schema}.dim_date').count()} linhas")
spark.table(f"{catalog}.{schema}.dim_date").limit(3).display()

In [0]:
# ============================================================
# 4. FACT_SALES
#    Tabela fato ligada às dimensões via surrogate keys
#    Joins: sales.customer_id → dim_customer.customer_id
#           sales.product_id  → dim_product.product_id
#           sales.order_date   → dim_date.date
# ============================================================

from pyspark.sql.functions import date_format, col, broadcast

# Carregar dimensões (pequenas, broadcast join)
dim_customer = spark.table(f"{catalog}.{schema}.dim_customer").select("customer_sk", "customer_id")
dim_product  = spark.table(f"{catalog}.{schema}.dim_product").select("product_sk", "product_id")
dim_date     = spark.table(f"{catalog}.{schema}.dim_date").select("date_sk", col("date").alias("order_date"))

# Carregar vendas da silver
df_sales = spark.table(f"{catalog}.silver.sales")

# Join com dim_customer
df_fact = (
    df_sales.alias("s")
    .join(broadcast(dim_customer).alias("dc"), col("s.customer_id") == col("dc.customer_id"), "left")
    .join(broadcast(dim_product).alias("dp"), col("s.product_id") == col("dp.product_id"), "left")
    .join(broadcast(dim_date).alias("dd"), col("s.order_date") == col("dd.order_date"), "left")
    .select(
        col("s.order_id"),
        col("dc.customer_sk"),
        col("dp.product_sk"),
        col("dd.date_sk"),
        col("s.order_date"),
        col("s.order_time"),
        col("s.delivery_date"),
        col("s.quantity"),
        col("s.unit_price"),
        col("s.order_value"),
        col("s.shipping_cost"),
        col("s.coupon_code"),
        col("s.coupon_discount"),
        col("s.total_amount"),
        col("s.payment_mode"),
        col("s.order_status"),
        col("s.rating"),
    )
)

(
    df_fact.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.fact_sales")
)

print(f"fact_sales criada: {spark.table(f'{catalog}.{schema}.fact_sales').count()} linhas")
spark.table(f"{catalog}.{schema}.fact_sales").limit(3).display()

In [0]:
# ============================================================
# 5. VALIDAÇÃO DO MODELO STAR
#    a) Chave natural única nas dimensões
#    b) Quantidade de linhas antes e depois dos joins
#    c) Clientes e produtos sem correspondência
#    d) Receita antes e depois da modelagem
# ============================================================

print("="*60)
print("  VALIDAÇÃO DO MODELO STAR SCHEMA")
print("="*60)

# --- a) Chave natural única ---
print("\n--- a) UNICIDADE DAS CHAVES NATURAIS ---")

for tbl, key in [("dim_customer", "customer_id"), ("dim_product", "product_id"), ("dim_date", "date_sk")]:
    total = spark.table(f"{catalog}.{schema}.{tbl}").count()
    distinct = spark.table(f"{catalog}.{schema}.{tbl}").select(key).distinct().count()
    status = "OK ✓" if total == distinct else f"DUPLICADOS ✗ ({total - distinct} duplicados)"
    print(f"  {tbl:20s} | {key:15s} | total={total:>8,} | distinct={distinct:>8,} | {status}")

# --- b) Linhas antes e depois dos joins ---
print("\n--- b) CONTAGEM DE LINHAS ---")

silver_sales_count = spark.table(f"{catalog}.silver.sales").count()
fact_sales_count   = spark.table(f"{catalog}.{schema}.fact_sales").count()

print(f"  silver.sales (antes):      {silver_sales_count:>8,} linhas")
print(f"  gold.fact_sales (depois): {fact_sales_count:>8,} linhas")
print(f"  Diferença:                 {silver_sales_count - fact_sales_count:>8,} linhas perdidas")

# --- c) Clientes e produtos sem correspondência ---
print("\n--- c) CLIENTES E PRODUTOS SEM CORRESPONDÊNCIA ---")

fact = spark.table(f"{catalog}.{schema}.fact_sales")

# Vendas com customer_sk NULL (cliente não encontrado na dimensão)
customers_missing = fact.filter(col("customer_sk").isNull()).count()
products_missing  = fact.filter(col("product_sk").isNull()).count()
dates_missing     = fact.filter(col("date_sk").isNull()).count()

print(f"  Vendas sem customer_sk (cliente não encontrado): {customers_missing:,}")
print(f"  Vendas sem product_sk  (produto não encontrado):  {products_missing:,}")
print(f"  Vendas sem date_sk     (data não encontrada):     {dates_missing:,}")

# Quantos customer_ids da silver.sales não existem em dim_customer
sales_customers = spark.table(f"{catalog}.silver.sales").select("customer_id").distinct()
dim_customers   = spark.table(f"{catalog}.{schema}.dim_customer").select("customer_id").distinct()
customers_not_in_dim = sales_customers.exceptAll(dim_customers).count()

sales_products = spark.table(f"{catalog}.silver.sales").select("product_id").distinct()
dim_products   = spark.table(f"{catalog}.{schema}.dim_product").select("product_id").distinct()
products_not_in_dim = sales_products.exceptAll(dim_products).count()

print(f"  customer_ids em sales mas não em dim_customer: {customers_not_in_dim:,}")
print(f"  product_ids  em sales mas não em dim_product:  {products_not_in_dim:,}")

# --- d) Receita antes e depois ---
print("\n--- d) RECEITA TOTAL: ANTES vs DEPOIS ---")

revenue_silver = spark.table(f"{catalog}.silver.sales").agg({"total_amount": "sum"}).collect()[0][0]
revenue_fact   = spark.table(f"{catalog}.{schema}.fact_sales").agg({"total_amount": "sum"}).collect()[0][0]

print(f"  Receita silver.sales: {revenue_silver:,.2f}")
print(f"  Receita gold.fact_sales: {revenue_fact:,.2f}")
print(f"  Diferença: {revenue_silver - revenue_fact:,.2f}")

if abs(revenue_silver - revenue_fact) < 0.01:
    print("  Status: RECEITA OK ✓ (sem perda)")
else:
    print("  Status: ATENÇÃO — há diferença na receita ✗")

print("\n" + "="*60)
print("  VALIDAÇÃO CONCLUÍDA")
print("="*60)

In [0]:
# ============================================================
# 6. CUSTOMER_SUMMARY_V2 (PySpark)
#    Recalcula o customer_summary usando a star schema
#    Não apaga o gold.customer_summary original
# ============================================================

from pyspark.sql.functions import (
    sum as spark_sum, count as spark_count, avg as spark_avg, 
    max as spark_max, countDistinct, col, round as spark_round
)
from pyspark.sql.window import Window

fact = spark.table(f"{catalog}.{schema}.fact_sales")
dim_customer = spark.table(f"{catalog}.{schema}.dim_customer")
dim_product = spark.table(f"{catalog}.{schema}.dim_product")

# Join fact + dim_customer + dim_product
df_joined = (
    fact.join(dim_customer, "customer_sk", "inner")
        .join(dim_product, "product_sk", "inner")
)

# Agregar por cliente
customer_summary_v2 = (
    df_joined.groupBy(
        dim_customer["customer_id"],
        dim_customer["customer_name"],
        dim_customer["customer_tier"],
        dim_customer["city"],
        dim_customer["state"],
    )
    .agg(
        spark_count("order_id").alias("total_orders"),
        spark_sum("total_amount").alias("total_spent"),
        spark_avg("total_amount").alias("avg_order_value"),
        spark_max("total_amount").alias("highest_order_value"),
        countDistinct("product_sk").alias("unique_products_bought"),
        countDistinct("order_date").alias("active_days"),
    )
)

# Adicionar colunas derivadas: categoria favorita e pagamento preferido
favorite_cat = (
    df_joined.groupBy("customer_id", "category")
    .agg(spark_count("*").alias("cnt"))
    .withColumn("rank", row_number().over(Window.partitionBy("customer_id").orderBy(col("cnt").desc())))
    .filter(col("rank") == 1)
    .select("customer_id", col("category").alias("favorite_category"))
)

preferred_pay = (
    df_joined.groupBy("customer_id", "payment_mode")
    .agg(spark_count("*").alias("cnt"))
    .withColumn("rank", row_number().over(Window.partitionBy("customer_id").orderBy(col("cnt").desc())))
    .filter(col("rank") == 1)
    .select("customer_id", col("payment_mode").alias("preferred_payment_mode"))
)

avg_rating = (
    df_joined.filter(col("rating").isNotNull())
    .groupBy("customer_id")
    .agg(spark_avg("rating").alias("avg_rating_given"))
)

customer_summary_v2 = (
    customer_summary_v2
    .join(favorite_cat, "customer_id", "left")
    .join(preferred_pay, "customer_id", "left")
    .join(avg_rating, "customer_id", "left")
    .select(
        "customer_id", "customer_name", "customer_tier", "city", "state",
        "total_orders", "total_spent", "avg_order_value", "highest_order_value",
        "unique_products_bought", "active_days",
        "favorite_category", "preferred_payment_mode", "avg_rating_given",
    )
)

(
    customer_summary_v2.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.customer_summary_v2")
)

print(f"customer_summary_v2 criada: {spark.table(f'{catalog}.{schema}.customer_summary_v2').count()} linhas")
spark.table(f"{catalog}.{schema}.customer_summary_v2").limit(5).display()

In [0]:
# ============================================================
# 7. PRODUCT_SUMMARY_V2 (PySpark)
#    Recalcula o product_summary usando a star schema
#    Não apaga o gold.product_summary original
# ============================================================

from pyspark.sql.functions import (
    sum as spark_sum, count as spark_count, avg as spark_avg,
    countDistinct, col
)

fact = spark.table(f"{catalog}.{schema}.fact_sales")
dim_customer = spark.table(f"{catalog}.{schema}.dim_customer")
dim_product = spark.table(f"{catalog}.{schema}.dim_product")

df_joined = (
    fact.join(dim_product, "product_sk", "inner")
        .join(dim_customer, "customer_sk", "inner")
)

product_summary_v2 = (
    df_joined.groupBy(
        dim_product["product_id"],
        dim_product["product_name"],
        dim_product["category"],
        dim_product["brand"],
        dim_product["selling_price"],
        dim_product["avg_rating"],
    )
    .agg(
        spark_count("order_id").alias("total_orders"),
        spark_sum("quantity").alias("total_units_sold"),
        spark_sum("total_amount").alias("total_revenue"),
        spark_avg("total_amount").alias("avg_order_value"),
        countDistinct("customer_sk").alias("unique_customers"),
        spark_avg("rating").alias("avg_customer_rating"),
    )
)

(
    product_summary_v2.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.product_summary_v2")
)

print(f"product_summary_v2 criada: {spark.table(f'{catalog}.{schema}.product_summary_v2').count()} linhas")
spark.table(f"{catalog}.{schema}.product_summary_v2").limit(5).display()